In [69]:
CONFIG = {
    'repo_url': 'https://github.com/CERN/TIGRE',
    'model_name': 'Qwen/Qwen2.5-7B-Instruct',
    'vllm_url': 'http://localhost:8001',
    'top_k': 5,
}

REPO_URL = CONFIG['repo_url']
MODEL_NAME = CONFIG['model_name']
VLLM_URL = CONFIG['vllm_url']
TOP_K = CONFIG['top_k']

In [70]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from urllib.parse import urlparse
from typing import Any

import numpy as np
import requests
from sentence_transformers import SentenceTransformer

## README

In [71]:
@dataclass
class RepoRef:
    provider: str  # github | gitlab
    owner: str
    repo: str


def parse_repo_url(repo_url: str) -> RepoRef:
    """
    Summary: Parse a repository URL into a structured reference.
    Parameters:
        repo_url (str): GitHub or GitLab repository URL.
    Returns:
        RepoRef: Parsed provider, owner, and repo fields.
    """
    parsed = urlparse(repo_url)
    host = parsed.netloc.lower()
    path_parts = [p for p in parsed.path.strip('/').split('/') if p]

    if len(path_parts) < 2:
        raise ValueError(f'Invalid repository URL: {repo_url}')

    owner, repo = path_parts[0], path_parts[1]
    if repo.endswith('.git'):
        repo = repo[:-4]

    if 'github.com' in host:
        return RepoRef(provider='github', owner=owner, repo=repo)
    if 'gitlab.com' in host:
        return RepoRef(provider='gitlab', owner=owner, repo=repo)

    raise ValueError('Only GitHub and GitLab URLs are supported.')


def candidate_readme_urls(ref: RepoRef) -> list[str]:
    """
    Summary: Build candidate raw README URLs.
    Parameters:
        ref (RepoRef): Parsed repository reference.
    Returns:
        list[str]: Ordered raw README URL candidates.
    """
    branches = ['main', 'master']
    filenames = ['README.md', 'README.MD', 'readme.md', 'README.rst', 'README.txt']

    urls: list[str] = []
    for branch in branches:
        for filename in filenames:
            if ref.provider == 'github':
                urls.append(f'https://raw.githubusercontent.com/{ref.owner}/{ref.repo}/{branch}/{filename}')
            else:
                urls.append(f'https://gitlab.com/{ref.owner}/{ref.repo}/-/raw/{branch}/{filename}')
    return urls


def fetch_readme(ref: RepoRef, timeout: int = 20) -> tuple[str, str]:
    """
    Summary: Fetch the repository README from the first working URL.
    Parameters:
        ref (RepoRef): Parsed repository reference.
        timeout (int): HTTP timeout in seconds.
    Returns:
        tuple[str, str]: README text and the source URL used.
    """
    headers = {'User-Agent': 'maSMP-ollama-readme-test/1.0'}

    for url in candidate_readme_urls(ref):
        try:
            resp = requests.get(url, headers=headers, timeout=timeout)
            if resp.status_code == 200 :
                return resp.text, url
        except Exception:
            continue

    raise RuntimeError('README not found on main/master with common README filenames.')

In [72]:
ref_preview = parse_repo_url(REPO_URL)
readme_text_preview, readme_url_preview = fetch_readme(ref_preview, timeout=20)

print(f'[README URL] {readme_url_preview}')
print(f'[README Length] {len(readme_text_preview)} chars')

[README URL] https://raw.githubusercontent.com/CERN/TIGRE/master/README.md
[README Length] 14111 chars


## Chunking

In [73]:
def split_with_metadata(md_text):
    """
    Summary: Split markdown into heading-aware sections.
    Parameters:
        md_text: Raw README markdown text.
    Returns:
        list[dict]: Section records with heading, level, and content.
    """
    lines = md_text.split("\n")
    chunks = []

    current_chunk = {"heading": None, "level": None, "content": []}

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        match = re.match(r'^(#{1,6})\s+(.*)', line)
        if match:
            if current_chunk["content"]:
                chunks.append(current_chunk)

            current_chunk = {
                "heading": match.group(2),
                "level": len(match.group(1)),
                "content": []
            }
            i += 1
            continue

        if i + 1 < len(lines):
            next_line = lines[i + 1].strip()

            if re.match(r'^=+$', next_line):
                if current_chunk["content"]:
                    chunks.append(current_chunk)

                current_chunk = {
                    "heading": line,
                    "level": 1,
                    "content": []
                }
                i += 2
                continue

            elif re.match(r'^-+$', next_line):
                if current_chunk["content"]:
                    chunks.append(current_chunk)

                current_chunk = {
                    "heading": line,
                    "level": 2,
                    "content": []
                }
                i += 2
                continue

        current_chunk["content"].append(lines[i])
        i += 1

    if current_chunk["content"]:
        chunks.append(current_chunk)

    return chunks


def hybrid_chunking(section, max_chars=1200, overlap=200):
    """
    Summary: Split oversized sections into overlapping chunks.
    Parameters:
        section (dict): Section record from split_with_metadata().
        max_chars (int): Maximum characters per chunk.
        overlap (int): Shared overlap between adjacent chunks.
    Returns:
        list[dict]: Chunk records with heading and content.
    """
    text = "\n".join(section["content"]).strip()

    if len(text) <= max_chars:
        return [{
            "heading": section["heading"],
            "content": text
        }]

    chunks = []
    start = 0

    while start < len(text):
        end = start + max_chars
        chunk_text = text[start:end]

        chunks.append({
            "heading": section["heading"],
            "content": chunk_text
        })

        start += max_chars - overlap

    return chunks

In [74]:
sections = split_with_metadata(readme_text_preview)
sections = [s for s in sections if s['heading'] is not None]

all_chunks = []
for section in sections:
    all_chunks.extend(hybrid_chunking(section))

print(f'Sections: {len(sections)} | Chunks: {len(all_chunks)}')

Sections: 10 | Chunks: 19


## Embedding + Retrieval

In [75]:
# LLM query + property hint + rule config
PROPERTY_QUERIES = {
    'license': ['license', 'licensing', 'copyright', 'spdx'],
    'installation': ['installation', 'install', 'pip', 'conda', 'requirements', 'setup'],
    'contact': ['contact', 'email', 'maintainer', 'author'],
    'contributors': [
        'contributors', 'contributor', 'authors', 'author', 'maintainers',
        'maintainer', 'team', 'credits', 'acknowledgements', 'thanks',
        'community', '@', 'github.com'
    ],
    'links': [
        'paper', 'publication', 'publications', 'cite', 'citation', 'arxiv',
        'doi', 'research', 'reference', 'further reading', 'docs', 'url', 'link', 'read the article'
    ],
    'description': [
        'overview', 'about', 'what is', 'purpose', 'project', 'repository',
        'toolkit', 'library', 'framework', 'package', 'goal', 'features',
        'introduction', 'summary', 'readme'
    ],
}

PROPERTY_SCHEMA_HINTS = {
    'license': 'normalized SPDX-style string if explicit, else null',
    'installation': 'short installation instruction summary, else null',
    'contact': 'email or contact link, else null',
    'contributors': 'list of contributors with name/github_url if available, else null',
    'links': 'list of relevant links as objects {title, url, is_working, status_code, relevance}, else null',
    'description': 'one or two concise lines describing what the repository does, else null',
}

PROPERTY_RULES = {
    'license': 'For license, return a normalized SPDX label and an exact evidence quote if present. ',
    'contributors': (
        'For contributors return STRICT JSON where value is a list of objects: '
        '[{"name": string, "github_url": string|null}]. Include ALL contributors found in provided chunks. '
    ),
    'links': (
        'For links return STRICT JSON where value is a list of objects: '
        '[{"title": string|null, "url": string, "relevance": "paper"|"docs"|"other"}]. '
        'Focus on relevant paper/publication links and documentation links found in provided chunks only. '
    ),
    'description': (
        'For description return brief factual lines about what the repository does, '
        'grounded only in the provided chunks. Avoid hype and guessing. '
    ),
    'default': 'For non-license fields, return a concise value, plus a supporting quote when available. ',
}

# Regex config
LICENSE_PATTERNS = [
    (r'bsd\s*[- ]?3\s*[- ]?clause|bsd-3-clause', 'BSD-3-Clause'),
    (r'bsd\s*[- ]?2\s*[- ]?clause|bsd-2-clause', 'BSD-2-Clause'),
    (r'\bmit\b(?:\s+license)?', 'MIT'),
    (r'apache\s*2\.0|apache-2\.0|apache license', 'Apache-2.0'),
    (r'gpl\s*v?3|gnu\s+general\s+public\s+license\s*v?3', 'GPL-3.0'),
    (r'gpl\s*v?2|gnu\s+general\s+public\s+license\s*v?2', 'GPL-2.0'),
    (r'lgpl\s*v?3|lesser\s+general\s+public\s+license\s*v?3', 'LGPL-3.0'),
    (r'mpl\s*2\.0|mozilla\s+public\s+license\s*2\.0', 'MPL-2.0'),
]

In [76]:
def _safe_text(x: Any) -> str:
    """
    Summary: Convert optional values into safe text.
    Parameters:
        x (Any): Input value to normalize.
    Returns:
        str: Input text or an empty string for None.
    """
    return '' if x is None else str(x)


def prepare_chunk_records(chunks: list[dict]) -> list[dict]:
    """
    Summary: Normalize chunks into retrieval-ready records.
    Parameters:
        chunks (list[dict]): Chunk list from hybrid_chunking().
    Returns:
        list[dict]: Indexed chunk records with full_text and metadata.
    """
    records = []
    for i, c in enumerate(chunks):
        heading = _safe_text(c.get('heading')).strip()
        content = _safe_text(c.get('content')).strip()
        if not content:
            continue
        records.append({
            'chunk_id': i,
            'heading': heading,
            'content': content,
            'char_len': len(content),
            'full_text': f"Heading: {heading}\n\n{content}" if heading else content,
        })
    return records


def build_retrieval_index(records: list[dict], model_name: str = 'sentence-transformers/all-MiniLM-L6-v2') -> dict:
    """
    Summary: Build an optional semantic embedding index.
    Parameters:
        records (list[dict]): Retrieval records from prepare_chunk_records().
        model_name (str): SentenceTransformer model name.
    Returns:
        dict: Index state with records, embeddings, and model metadata.
    """
    index = {
        'records': records,
        'embeddings': None,
        'model': None,
        'embedding_enabled': False,
        'model_name': model_name,
    }
    try:
        model = SentenceTransformer(model_name)
        vectors = model.encode([r['full_text'] for r in records], normalize_embeddings=True)
        index['embeddings'] = np.asarray(vectors, dtype=np.float32)
        index['model'] = model
        index['embedding_enabled'] = True
        print(f'[Embeddings] enabled ({model_name})')
    except Exception as exc:
        print(f'[Embeddings] disabled, lexical fallback only: {exc}')
    return index


def keyword_score(record: dict, query_terms: list[str]) -> float:
    """
    Summary: Score a chunk using lexical term overlap.
    Parameters:
        record (dict): Retrieval record with heading and content fields.
        query_terms (list[str]): Property query terms.
    Returns:
        float: Weighted lexical relevance score.
    """
    heading = record['heading'].lower()
    content = record['content'].lower()
    score = 0.0
    for term in query_terms:
        t = term.lower()
        if t in heading:
            score += 2.0
        if t in content:
            score += 1.0
    return score


def retrieve_top_chunks(index: dict, property_name: str, top_k: int = 5, alpha: float = 0.75) -> list[dict]:
    """
    Summary: Rank chunks for a property query.
    Parameters:
        index (dict): Retrieval index from build_retrieval_index().
        property_name (str): Target property name.
        top_k (int): Number of chunks to return.
        alpha (float): Blend weight for semantic versus lexical score.
    Returns:
        list[dict]: Top-ranked chunk records with score and rank.
    """
    terms = PROPERTY_QUERIES[property_name]
    records = index['records']

    kw = np.array([keyword_score(r, terms) for r in records], dtype=np.float32)
    if kw.max() > 0:
        kw = kw / kw.max()

    if index['embedding_enabled']:
        q = ' '.join(terms)
        qv = index['model'].encode([q], normalize_embeddings=True)[0]
        ev = index['embeddings']
        sem = ev @ np.asarray(qv, dtype=np.float32)
        sem = (sem + 1.0) / 2.0
        scores = alpha * sem + (1 - alpha) * kw
    else:
        scores = kw

    order = np.argsort(-scores)[:top_k]
    out = []
    for rank, idx in enumerate(order, start=1):
        r = dict(records[int(idx)])
        r['score'] = float(scores[int(idx)])
        r['rank'] = rank
        out.append(r)
    return out

In [77]:
chunk_records = prepare_chunk_records(all_chunks)
retrieval_index = build_retrieval_index(chunk_records)
print(f'[Chunk Records] {len(chunk_records)} usable chunks')

[Embeddings] enabled (sentence-transformers/all-MiniLM-L6-v2)
[Chunk Records] 19 usable chunks


## Final Extraction Functions + Execution

### Ollama client utilities

In [78]:
def check_vllm_ready(model: str, vllm_url: str = 'http://localhost:8000', timeout: int = 10) -> tuple[bool, str]:
    endpoint = vllm_url.rstrip('/') + '/v1/models'
    try:
        resp = requests.get(endpoint, timeout=timeout)
    except requests.exceptions.RequestException as exc:
        return False, f'Cannot reach vLLM at {endpoint}: {exc}'

    if resp.status_code != 200:
        return False, f'vLLM responded with {resp.status_code}: {resp.text}'

    data = resp.json()
    model_names = [m.get('id', '') for m in data.get('data', [])]

    if model in model_names:
        return True, f'vLLM is ready. Model `{model}` found.'
    if not model_names:
        return False, 'vLLM is running but no models are exposed.'
    return False, f'Model `{model}` not found. Available: {model_names}'

def run_vllm(prompt: str, model: str, vllm_url: str = 'http://localhost:8000', timeout: int = 120) -> str:
    endpoint = vllm_url.rstrip('/') + '/v1/chat/completions'
    payload = {
        'model': model,
        'messages': [
            {'role': 'system', 'content': 'Return valid JSON only.'},
            {'role': 'user', 'content': prompt},
        ],
        'temperature': 0,
        'top_p': 0.7,
        'max_tokens': 700,
    }

    try:
        resp = requests.post(endpoint, json=payload, timeout=timeout)
    except requests.exceptions.RequestException as exc:
        raise RuntimeError(f'Connection failed to {endpoint}. Ensure vLLM is running.') from exc

    if resp.status_code != 200:
        raise RuntimeError(f'vLLM error {resp.status_code}: {resp.text}')

    data = resp.json()
    try:
        return data['choices'][0]['message']['content']
    except (KeyError, IndexError, TypeError) as exc:
        raise RuntimeError(f'Unexpected vLLM response: {data}') from exc

### Model output parsing

In [79]:
def extract_json(text: str) -> dict:
    """
    Summary: Extract a JSON object from model output.
    Parameters:
        text (str): Raw model output text.
    Returns:
        dict: Parsed JSON object.
    """
    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r'\{[\s\S]*\}', text)
    if not match:
        raise ValueError('Model output does not contain JSON.')

    return json.loads(match.group(0))


def extract_html(text: str) -> str:
    """
    Summary: Extract an HTML snippet from model output.
    Parameters:
        text (str): Raw model output text.
    Returns:
        str: Best HTML fragment found in the text.
    """
    text = text.strip()

    if re.search(r'<\s*html\b', text, flags=re.IGNORECASE):
        return text

    block_match = re.search(
        r'(<(table|ul|ol|div|section|article|p)[^>]*>[\s\S]*?</\2>)',
        text,
        flags=re.IGNORECASE,
    )
    if block_match:
        return block_match.group(1)

    generic_match = re.search(r'(<([a-zA-Z][a-zA-Z0-9]*)[^>]*>[\s\S]*?</\2>)', text)
    if generic_match:
        return generic_match.group(1)

    raise ValueError('Model output does not contain HTML.')

### Regex + entity extraction helpers

In [80]:
def extract_license_from_readme(readme_text: str) -> tuple[str | None, str | None]:
    """
    Summary: Extract a normalized license label from README text.
    Parameters:
        readme_text (str): README content to scan.
    Returns:
        tuple[str | None, str | None]: SPDX-like label and supporting evidence quote.
    """
    patterns = LICENSE_PATTERNS

    low = readme_text.lower()
    for pat, spdx in patterns:
        m = re.search(pat, low, flags=re.IGNORECASE)
        if m:
            start = max(0, m.start() - 80)
            end = min(len(readme_text), m.end() + 80)
            evidence = readme_text[start:end].replace('\n', ' ').strip()
            return spdx, evidence

    return None, None


def _strip_html_tags(text: str) -> str:
    """
    Summary: Remove HTML tags from text.
    Parameters:
        text (str): HTML-containing string.
    Returns:
        str: Plain-text value without tags.
    """
    return re.sub(r'<[^>]+>', '', text or '').strip()


def parse_contributor_links_from_html(html_text: str) -> list[dict]:
    """
    Summary: Extract contributor links from HTML.
    Parameters:
        html_text (str): HTML snippet containing contributor links.
    Returns:
        list[dict]: Contributor link records with name and GitHub URL.
    """
    links: list[dict] = []
    seen_urls: set[str] = set()
    for href, label in re.findall(
        r'<a\s+[^>]*href=["\']([^"\']+)["\'][^>]*>([\s\S]*?)</a>',
        html_text,
        flags=re.IGNORECASE,
    ):
        if 'github.com/' not in href.lower():
            continue
        url = href.strip()
        if not url or url in seen_urls:
            continue
        seen_urls.add(url)
        name = _strip_html_tags(label)
        if not name:
            name = url.rstrip('/').split('/')[-1]
        links.append({'name': name, 'github_url': url})
    return links


def extract_contributors_from_text(text: str) -> list[dict]:
    """
    Summary: Extract contributor candidates from plain text.
    Parameters:
        text (str): Text that may contain contributor references.
    Returns:
        list[dict]: Normalized contributor link records.
    """
    found: list[dict] = []
    seen_urls: set[str] = set()

    for name, url in re.findall(
        r'\[([^\]]+)\]\((https?://(?:www\.)?github\.com/[A-Za-z0-9_.-]+(?:/[A-Za-z0-9_.-]+)?)\)',
        text,
        flags=re.IGNORECASE,
    ):
        u = url.strip()
        if u and u not in seen_urls:
            seen_urls.add(u)
            found.append({'name': name.strip(), 'github_url': u})

    for url in re.findall(r'https?://(?:www\.)?github\.com/[A-Za-z0-9_.-]+(?:/[A-Za-z0-9_.-]+)?', text, flags=re.IGNORECASE):
        u = url.strip().rstrip(').,;')
        if u and u not in seen_urls:
            seen_urls.add(u)
            name = u.rstrip('/').split('/')[-1]
            found.append({'name': name, 'github_url': u})

    for handle in re.findall(r'(?<![\w/])@([A-Za-z0-9-]{1,39})\b', text):
        u = f'https://github.com/{handle}'
        if u not in seen_urls:
            seen_urls.add(u)
            found.append({'name': handle, 'github_url': u})

    return found


def _normalize_contributor_list(items: list[dict]) -> list[dict]:
    """
    Summary: Normalize contributor records.
    Parameters:
        items (list[dict]): Candidate contributor records.
    Returns:
        list[dict]: Deduplicated contributor records with name and GitHub URL.
    """
    merged: list[dict] = []
    seen_urls: set[str] = set()
    for item in items:
        if not isinstance(item, dict):
            continue
        url = str(item.get('github_url') or '').strip()
        name = str(item.get('name') or '').strip()
        if not url:
            continue
        if 'github.com/' not in url.lower():
            continue
        if url in seen_urls:
            continue
        if not name:
            name = url.rstrip('/').split('/')[-1]
        seen_urls.add(url)
        merged.append({'name': name, 'github_url': url})
    return merged


def _contributors_from_llm_json(raw: dict) -> list[dict]:
    """
    Summary: Parse contributor JSON produced by the model.
    Parameters:
        raw (dict): Parsed model JSON output.
    Returns:
        list[dict]: Normalized contributor records.
    """
    value = raw.get('value')
    items: list[dict] = []

    if isinstance(value, list):
        for v in value:
            if isinstance(v, dict):
                items.append({'name': v.get('name'), 'github_url': v.get('github_url')})
            elif isinstance(v, str):
                items.extend(extract_contributors_from_text(v))
    elif isinstance(value, dict):
        if 'contributors' in value and isinstance(value['contributors'], list):
            for v in value['contributors']:
                if isinstance(v, dict):
                    items.append({'name': v.get('name'), 'github_url': v.get('github_url')})
                elif isinstance(v, str):
                    items.extend(extract_contributors_from_text(v))
        else:
            items.append({'name': value.get('name'), 'github_url': value.get('github_url')})
    elif isinstance(value, str):
        items.extend(extract_contributors_from_text(value))

    for extra_field in ('evidence_quote', 'source_heading'):
        extra_val = raw.get(extra_field)
        if isinstance(extra_val, str):
            items.extend(extract_contributors_from_text(extra_val))

    return _normalize_contributor_list(items)


def extract_links_from_text(text: str) -> list[dict]:
    """
    Summary: Extract link candidates from plain text.
    Parameters:
        text (str): Text that may contain markdown links or bare URLs.
    Returns:
        list[dict]: Link candidates with title and URL.
    """
    found: list[dict] = []
    seen: set[str] = set()

    for title, url in re.findall(r'\[([^\]]+)\]\((https?://[^\s)]+)\)', text, flags=re.IGNORECASE):
        u = url.strip().rstrip(').,;')
        if u and u not in seen:
            seen.add(u)
            found.append({'title': title.strip(), 'url': u})

    for url in re.findall(r'https?://[^\s<>()\]"\']+', text, flags=re.IGNORECASE):
        u = url.strip().rstrip(').,;')
        if u and u not in seen:
            seen.add(u)
            found.append({'title': None, 'url': u})

    return found


def _is_relevant_paper_link(url: str, title: str | None = None, context: str | None = None) -> bool:
    """
    Summary: Check whether a link is publication-related.
    Parameters:
        url (str): Candidate URL.
        title (str | None): Optional link title.
        context (str | None): Surrounding text for relevance clues.
    Returns:
        bool: True when the link looks paper- or publication-related.
    """
    u = (url or '').lower()
    t = (title or '').lower()
    c = (context or '').lower()

    paper_domains = [
        'arxiv.org', 'doi.org', 'paperswithcode.com', 'openreview.net',
        'ieeexplore.ieee.org', 'dl.acm.org', 'springer.com', 'nature.com',
        'sciencedirect.com', 'biorxiv.org', 'medrxiv.org'
    ]
    if any(d in u for d in paper_domains):
        return True

    paper_terms = ['paper', 'publication', 'publications', 'cite', 'citation', 'preprint', 'manuscript']
    return any(term in u for term in paper_terms) or any(term in t for term in paper_terms) or any(term in c for term in paper_terms)


def check_link_status(url: str, timeout: int = 8) -> dict:
    """
    Summary: Probe a link and return its status metadata.
    Parameters:
        url (str): URL to validate.
        timeout (int): HTTP timeout in seconds.
    Returns:
        dict: Working flag, status code, final URL, and error details.
    """
    try:
        head = requests.head(
            url,
            timeout=timeout,
            allow_redirects=True,
            headers={'User-Agent': 'maSMP-link-check/1.0'},
        )
        status = int(head.status_code)
        if status >= 400 or status == 405:
            get_resp = requests.get(
                url,
                timeout=timeout,
                allow_redirects=True,
                headers={'User-Agent': 'maSMP-link-check/1.0'},
                stream=True,
            )
            status = int(get_resp.status_code)
            final_url = str(get_resp.url)
            get_resp.close()
        else:
            final_url = str(head.url)

        return {
            'is_working': 200 <= status < 400,
            'status_code': status,
            'final_url': final_url,
            'error': None,
        }
    except Exception as exc:
        return {
            'is_working': False,
            'status_code': None,
            'final_url': None,
            'error': str(exc),
        }


def _normalize_link_list(items: list[dict], relevance_default: str = 'other') -> list[dict]:
    """
    Summary: Normalize and deduplicate link records.
    Parameters:
        items (list[dict]): Candidate link records.
        relevance_default (str): Default relevance label.
    Returns:
        list[dict]: Standardized link records.
    """
    merged: list[dict] = []
    seen: set[str] = set()
    for item in items:
        if not isinstance(item, dict):
            continue
        url = str(item.get('url') or '').strip().rstrip(').,;')
        if not url or not re.match(r'^https?://', url, flags=re.IGNORECASE):
            continue
        if url in seen:
            continue
        seen.add(url)
        merged.append({
            'title': (str(item.get('title')).strip() if item.get('title') is not None else None),
            'url': url,
            'relevance': str(item.get('relevance') or relevance_default),
            'is_working': item.get('is_working'),
            'status_code': item.get('status_code'),
            'final_url': item.get('final_url'),
            'error': item.get('error'),
        })
    return merged


def _links_from_llm_json(raw: dict) -> list[dict]:
    """
    Summary: Parse link JSON produced by the model.
    Parameters:
        raw (dict): Parsed model JSON output.
    Returns:
        list[dict]: Normalized link records.
    """
    value = raw.get('value')
    items: list[dict] = []

    if isinstance(value, list):
        for v in value:
            if isinstance(v, dict):
                items.append({
                    'title': v.get('title') or v.get('name'),
                    'url': v.get('url') or v.get('link') or v.get('href'),
                    'relevance': v.get('relevance') or 'other',
                })
            elif isinstance(v, str):
                for i in extract_links_from_text(v):
                    items.append({'title': i.get('title'), 'url': i.get('url'), 'relevance': 'other'})
    elif isinstance(value, dict):
        items.append({
            'title': value.get('title') or value.get('name'),
            'url': value.get('url') or value.get('link') or value.get('href'),
            'relevance': value.get('relevance') or 'other',
        })
    elif isinstance(value, str):
        for i in extract_links_from_text(value):
            items.append({'title': i.get('title'), 'url': i.get('url'), 'relevance': 'other'})

    return _normalize_link_list(items, relevance_default='other')

### Description and prompt helpers

In [81]:
def _normalize_description_text(text: str | None) -> str | None:
    """
    Summary: Normalize free-text descriptions for display and prompts.
    Parameters:
        text (str | None): Candidate description text.
    Returns:
        str | None: Cleaned description or None when empty.
    """
    if not isinstance(text, str):
        return None
    compact = ' '.join(text.strip().split())
    if not compact:
        return None
    if len(compact) > 260:
        compact = compact[:257].rstrip() + '...';
    return compact


def _extract_installation_summary_from_chunks(chunks: list[dict]) -> str | None:
    """
    Summary: Derive a short installation summary from retrieved chunks.
    Parameters:
        chunks (list[dict]): Retrieved evidence chunks.
    Returns:
        str | None: Short install summary when available.
    """
    command_lines: list[str] = []
    seen: set[str] = set()
    for c in chunks:
        text = f"{c.get('heading','')}\n{c.get('content','')}"
        for line in text.splitlines():
            s = line.strip().strip('`')
            if not s:
                continue
            if re.search(r'(^|\s)(pip(3)?\s+install|conda\s+install|poetry\s+add|uv\s+pip\s+install|python\s+-m\s+pip\s+install|git\s+clone|cmake\s+|make\b|docker\s+build|docker\s+run)(\s|$)', s, flags=re.IGNORECASE):
                if s not in seen:
                    seen.add(s)
                    command_lines.append(s)

    if command_lines:
        return _normalize_description_text('Install via: ' + '; '.join(command_lines[:2]))

    merged = '\n'.join([f"{c.get('heading','')}\n{c.get('content','')}" for c in chunks])
    m = re.search(r'(installation|getting started)[\s\S]{0,500}', merged, flags=re.IGNORECASE)
    if m:
        snippet = m.group(0).replace('\n', ' ')
        return _normalize_description_text(snippet)
    return None


def _extract_description_evidence_quote_from_chunks(chunks: list[dict], description_value: str | None = None) -> str | None:
    """
    Summary: Select a README sentence that supports the description.
    Parameters:
        chunks (list[dict]): Retrieved evidence chunks for the property.
        description_value (str | None): Candidate description text from the model.
    Returns:
        str | None: Best matching evidence sentence or None.
    """
    if not chunks:
        return None

    stopwords = {
        'with', 'that', 'this', 'from', 'using', 'fast', 'accurate', 'open',
        'source', 'toolbox', 'library', 'framework', 'package', 'project',
        'repository', 'tool', 'tools', 'for', 'the', 'and', 'to', 'of'
    }

    value_terms: list[str] = []
    if isinstance(description_value, str) and description_value.strip():
        value_terms = [
            term for term in re.findall(r'[A-Za-z]{4,}', description_value.lower())
            if term not in stopwords
        ]

    sentences: list[str] = []
    for c in chunks:
        combined = f"{c.get('heading', '')}\n{c.get('content', '')}"
        for sentence in re.split(r'(?<=[.!?])\s+|\n+', combined):
            sentence = sentence.strip()
            if sentence:
                sentences.append(sentence)

    if value_terms:
        for sentence in sentences:
            lower = sentence.lower()
            overlap = sum(1 for term in value_terms[:8] if term in lower)
            if overlap >= 2 or any(term in lower for term in value_terms[:4]):
                return _normalize_description_text(sentence)

    for sentence in sentences:
        lower = sentence.lower()
        if any(term in lower for term in PROPERTY_QUERIES.get('description', [])):
            return _normalize_description_text(sentence)

    return _normalize_description_text(sentences[0]) if sentences else None


def build_property_prompt(repo_url: str, property_name: str, chunks: list[dict]) -> str:
    """
    Summary: Build the property-specific prompt for the model.
    Parameters:
        repo_url (str): Repository URL being processed.
        property_name (str): Target property to extract.
        chunks (list[dict]): Retrieved evidence chunks.
    Returns:
        str: Structured prompt with schema and evidence context.
    """
    schema_hint = PROPERTY_SCHEMA_HINTS.get(property_name, 'string or null')
    property_rule = PROPERTY_RULES.get(property_name, PROPERTY_RULES['default'])

    context = []
    for c in chunks:
        context.append(
            f"[chunk_id={c['chunk_id']}; rank={c['rank']}; heading={c['heading']}]\n{c['content']}"
        )
    context_text = '\n\n-----\n\n'.join(context)

    return (
        f"Task: Extract property '{property_name}' from README evidence only. "
        f"{property_rule}"
        "No guessing. Return minified JSON only. "
        "If evidence is missing, set value=null, evidence_quote=null, source_heading=null, evidence_type='llm_assumption'. "
        "If evidence exists, evidence_quote should be an exact quote from the provided chunks whenever possible. "
        f"Schema: {{\"repository_url\": string, \"property\": string, \"value\": {schema_hint}, \"evidence_quote\": string|null, \"source_heading\": string|null, \"evidence_type\": \"readme_evidence\"|\"llm_assumption\", \"confidence\": number}}. "
        f"Repository URL: {repo_url}\n\n"
        f"Retrieved README chunks:\n{context_text}"
    )

### Validation and property orchestration

In [82]:
def validate_property_output(raw: dict, property_name: str, chunks: list[dict], repo_url: str) -> dict:
    """
    Summary: Validate and normalize the model output for one property.
    Parameters:
        raw (dict): Parsed model output.
        property_name (str): Target property name.
        chunks (list[dict]): Retrieved evidence chunks.
        repo_url (str): Repository URL being processed.
    Returns:
        dict: Final normalized property payload with evidence metadata.
    """
    merged_text = '\n'.join([c['content'] for c in chunks]).lower()
    headings = [str(c.get('heading') or '').strip().lower() for c in chunks if c.get('heading')]
    property_terms = PROPERTY_QUERIES.get(property_name, [])
    property_support = any(term.lower() in merged_text for term in property_terms)

    out = {
        'repository_url': repo_url,
        'property': property_name,
        'value': None,
        'evidence_quote': None,
        'source_heading': None,
        'evidence_type': 'llm_assumption',
        'confidence': 0.0,
    }
    if not isinstance(raw, dict):
        return out

    out['repository_url'] = str(raw.get('repository_url') or repo_url).strip()
    out['property'] = str(raw.get('property') or property_name).strip()
    out['value'] = raw.get('value')
    out['confidence'] = float(raw.get('confidence') or 0.0)

    raw_heading = raw.get('source_heading')
    heading_support = isinstance(raw_heading, str) and raw_heading.strip().lower() in headings
    if heading_support:
        out['source_heading'] = str(raw_heading).strip()

    quote = raw.get('evidence_quote')
    quote_text = quote.strip() if isinstance(quote, str) and quote.strip() else None
    has_exact_quote = bool(quote_text and quote_text.lower() in merged_text)

    if property_name == 'license':
        if has_exact_quote and out['value'] is not None:
            out['evidence_quote'] = quote_text
            out['evidence_type'] = 'readme_evidence'
            out['confidence'] = max(out['confidence'], 0.6)
        else:
            out['value'] = None
            out['evidence_quote'] = None
            out['source_heading'] = None
            out['evidence_type'] = 'llm_assumption'
            out['confidence'] = min(out['confidence'], 0.3)
        return out

    if property_name == 'contributors':
        contribs = _normalize_contributor_list(out['value'] if isinstance(out['value'], list) else [])
        out['value'] = contribs if contribs else None
        if out['value'] is None:
            out['evidence_type'] = 'llm_assumption'
            out['confidence'] = min(out['confidence'], 0.3)
            out['evidence_quote'] = None
            out['source_heading'] = None
        else:
            out['evidence_type'] = 'readme_evidence'
            out['confidence'] = max(out['confidence'], 0.7)
            if not quote_text:
                out['evidence_quote'] = None
        return out

    if property_name == 'links':
        links = _normalize_link_list(out['value'] if isinstance(out['value'], list) else [])
        out['value'] = links if links else None
        if out['value'] is None:
            out['evidence_type'] = 'llm_assumption'
            out['confidence'] = min(out['confidence'], 0.3)
            out['evidence_quote'] = None
            out['source_heading'] = None
        else:
            out['evidence_type'] = 'readme_evidence'
            out['confidence'] = max(out['confidence'], 0.7)
            if not quote_text:
                out['evidence_quote'] = None
        return out

    if property_name == 'description':
        out['value'] = _normalize_description_text(out['value'])
        if out['value'] is None:
            out['evidence_type'] = 'llm_assumption'
            out['confidence'] = min(out['confidence'], 0.3)
            out['evidence_quote'] = None
            out['source_heading'] = None
        else:
            out['evidence_type'] = 'readme_evidence' if (has_exact_quote or heading_support or property_support) else 'llm_assumption'
            out['confidence'] = max(out['confidence'], 0.65 if out['evidence_type'] == 'readme_evidence' else 0.4)
            if out['evidence_type'] == 'llm_assumption':
                out['evidence_quote'] = None
                out['source_heading'] = None
            elif not quote_text:
                out['evidence_quote'] = None
        return out

    if out['value'] in (None, '', []):
        out['evidence_quote'] = None
        out['source_heading'] = None
        out['evidence_type'] = 'llm_assumption'
        out['confidence'] = min(out['confidence'], 0.3)
        return out

    if has_exact_quote:
        out['evidence_quote'] = quote_text
        out['evidence_type'] = 'readme_evidence'
        out['confidence'] = max(out['confidence'], 0.65)
    elif heading_support or property_support:
        out['evidence_quote'] = quote_text
        out['evidence_type'] = 'readme_evidence'
        out['confidence'] = max(out['confidence'], 0.5)
    else:
        out['evidence_quote'] = None
        out['source_heading'] = None
        out['evidence_type'] = 'llm_assumption'
        out['confidence'] = min(out['confidence'], 0.4)

    return out


def extract_property_with_retrieval(repo_url: str, property_name: str, model: str, vllm_url: str, top_k: int = 5) -> dict:
    """
    Summary: Orchestrate retrieval, model inference, and validation for one property.
    Parameters:
        repo_url (str): Repository URL being processed.
        property_name (str): Target property name.
        model (str): Ollama model name.
        vllm_url (str): Base Ollama server URL.
        top_k (int): Maximum number of retrieved chunks.
    Returns:
        dict: Final property output with evidence and retrieval metadata.
    """
    if property_name == 'license':
        k, alpha = 4, 0.8
    elif property_name == 'contributors':
        k, alpha = max(top_k, 20), 0.65
    elif property_name == 'links':
        k, alpha = max(top_k, 12), 0.70
    elif property_name == 'description':
        k, alpha = max(top_k, 6), 0.75
    else:
        k, alpha = top_k, 0.75

    picked = retrieve_top_chunks(retrieval_index, property_name=property_name, top_k=k, alpha=alpha)
    prompt = build_property_prompt(repo_url=repo_url, property_name=property_name, chunks=picked)
    model_output = run_vllm(prompt=prompt, model=model, vllm_url=vllm_url, timeout=420)

    if property_name == 'contributors':
        seed_items: list[dict] = []
        for c in picked:
            seed_items.extend(extract_contributors_from_text(f"{c.get('heading','')}\n{c.get('content','')}"))
        seed_items = _normalize_contributor_list(seed_items)

        llm_items: list[dict] = []
        try:
            raw = extract_json(model_output)
            llm_items = _contributors_from_llm_json(raw)
        except ValueError:
            try:
                html_snippet = extract_html(model_output)
                llm_items = parse_contributor_links_from_html(html_snippet)
            except ValueError:
                llm_items = extract_contributors_from_text(model_output)

        merged_items = _normalize_contributor_list(seed_items + llm_items)
        raw = {
            'repository_url': repo_url,
            'property': 'contributors',
            'value': merged_items if merged_items else None,
            'evidence_quote': None,
            'source_heading': picked[0]['heading'] if picked else None,
            'evidence_type': 'readme_evidence' if merged_items else 'llm_assumption',
            'confidence': 0.75 if merged_items else 0.2,
        }
    elif property_name == 'links':
        seed_items: list[dict] = []
        for c in picked:
            combined = f"{c.get('heading','')}\n{c.get('content','')}"
            for item in extract_links_from_text(combined):
                relevance = 'paper' if _is_relevant_paper_link(item.get('url', ''), item.get('title'), combined) else 'other'
                if relevance == 'paper' or 'doc' in combined.lower() or 'readthedocs' in item.get('url', '').lower():
                    seed_items.append({
                        'title': item.get('title'),
                        'url': item.get('url'),
                        'relevance': 'paper' if relevance == 'paper' else 'docs',
                    })

        llm_items: list[dict] = []
        try:
            raw = extract_json(model_output)
            llm_items = _links_from_llm_json(raw)
        except ValueError:
            for item in extract_links_from_text(model_output):
                llm_items.append({
                    'title': item.get('title'),
                    'url': item.get('url'),
                    'relevance': 'paper' if _is_relevant_paper_link(item.get('url', ''), item.get('title'), model_output) else 'other',
                })

        merged_items = _normalize_link_list(seed_items + llm_items)

        checked_items: list[dict] = []
        for item in merged_items[:30]:
            status = check_link_status(item['url'])
            checked_items.append({
                **item,
                'is_working': status['is_working'],
                'status_code': status['status_code'],
                'final_url': status['final_url'],
                'error': status['error'],
            })

        raw = {
            'repository_url': repo_url,
            'property': 'links',
            'value': checked_items if checked_items else None,
            'evidence_quote': None,
            'source_heading': picked[0]['heading'] if picked else None,
            'evidence_type': 'readme_evidence' if checked_items else 'llm_assumption',
            'confidence': 0.75 if checked_items else 0.2,
        }
    elif property_name == 'installation':
        try:
            raw = extract_json(model_output)
        except ValueError:
            raw = {
                'repository_url': repo_url,
                'property': 'installation',
                'value': _extract_installation_summary_from_chunks(picked),
                'evidence_quote': None,
                'source_heading': picked[0]['heading'] if picked else None,
                'evidence_type': 'readme_evidence' if picked else 'llm_assumption',
                'confidence': 0.55 if picked else 0.1,
            }
        if not raw.get('value'):
            raw['value'] = _extract_installation_summary_from_chunks(picked)
            if raw['value']:
                raw['evidence_type'] = 'readme_evidence'
                raw['confidence'] = max(float(raw.get('confidence') or 0.0), 0.55)
                if not raw.get('source_heading') and picked:
                    raw['source_heading'] = picked[0].get('heading')
    elif property_name == 'description':
        try:
            raw = extract_json(model_output)
        except ValueError:
            raw = {
                'repository_url': repo_url,
                'property': 'description',
                'value': _normalize_description_text(model_output),
                'evidence_quote': None,
                'source_heading': picked[0]['heading'] if picked else None,
                'evidence_type': 'readme_evidence' if model_output else 'llm_assumption',
                'confidence': 0.45 if model_output else 0.1,
            }
    else:
        raw = extract_json(model_output)

    validated = validate_property_output(raw=raw, property_name=property_name, chunks=picked, repo_url=repo_url)
    if property_name == 'description' and isinstance(validated, dict):
        if not validated.get('evidence_quote') and validated.get('value'):
            evidence_quote = _extract_description_evidence_quote_from_chunks(picked, validated.get('value'))
            if evidence_quote:
                validated['evidence_quote'] = evidence_quote
                validated['evidence_type'] = 'readme_evidence'
                validated['confidence'] = max(float(validated.get('confidence') or 0.0), 0.6)
    validated['retrieved_chunk_ids'] = [c['chunk_id'] for c in picked]
    validated['retrieved_headings'] = [c['heading'] for c in picked]
    validated['retrieved_scores'] = [c.get('score') for c in picked]
    return validated

In [83]:
ok, msg = check_vllm_ready(model=MODEL_NAME, vllm_url=VLLM_URL, timeout=10)
print(msg)

properties_to_extract = ['license', 'installation', 'contact', 'contributors', 'links', 'description']
results = {}

for property_name in properties_to_extract:
    try:
        results[property_name] = extract_property_with_retrieval(
            repo_url=REPO_URL,
            property_name=property_name,
            model=MODEL_NAME,
            vllm_url=VLLM_URL,
            top_k=TOP_K,
        )
    except Exception as exc:
        results[property_name] = {'property': property_name, 'error': str(exc)}

license_rule, license_evidence = extract_license_from_readme(readme_text_preview)
if license_rule and isinstance(results.get('license'), dict):
    results['license']['value'] = license_rule
    results['license']['evidence_quote'] = license_evidence
    results['license']['source_heading'] = 'regex_rule'
    results['license']['evidence_type'] = 'readme_rule'
    results['license']['confidence'] = max(float(results['license'].get('confidence') or 0.0), 0.95)

description_result = results.get('description')
if isinstance(description_result, dict):
    if not description_result.get('evidence_quote'):
        description_result['evidence_quote'] = _extract_description_evidence_quote_from_chunks(
            all_chunks,
            description_result.get('value'),
        )
    if description_result.get('evidence_quote'):
        description_result['evidence_type'] = 'readme_evidence'
        description_result['confidence'] = max(float(description_result.get('confidence') or 0.0), 0.6)

print(json.dumps(results, indent=2, ensure_ascii=False))

vLLM is ready. Model `Qwen/Qwen2.5-7B-Instruct` found.
{
  "license": {
    "repository_url": "https://github.com/CERN/TIGRE",
    "property": "license",
    "value": "BSD-3-Clause",
    "evidence_quote": "It is released under the BSD License, meaning you can use and modify the software freely.",
    "source_heading": "Licensing",
    "evidence_type": "readme_evidence",
    "confidence": 1.0,
    "retrieved_chunk_ids": [
      10,
      11,
      18,
      1
    ],
    "retrieved_headings": [
      "Licensing",
      "Licensing",
      "Contributors",
      "TIGRE: Tomographic Iterative GPU-based Reconstruction Toolbox"
    ],
    "retrieved_scores": [
      0.6843735575675964,
      0.5801926255226135,
      0.5359349250793457,
      0.5181072354316711
    ]
  },
  "installation": {
    "repository_url": "https://github.com/CERN/TIGRE",
    "property": "installation",
    "value": "MATLAB and Python builds are both fully supported.",
    "evidence_quote": "- [Installation instructions

## Individual property extraction

In [84]:
def run_single_property(property_name: str) -> dict:
    """
    Summary: Extract and post-process one property using the shared pipeline.
    Parameters:
        property_name (str): Property name to extract.
    Returns:
        dict: Normalized extraction result for the property.
    """
    result = extract_property_with_retrieval(
        repo_url=REPO_URL,
        property_name=property_name,
        model=MODEL_NAME,
        vllm_url=VLLM_URL,
        top_k=TOP_K,
    )

    if property_name == 'license' and isinstance(result, dict):
        license_rule, license_evidence = extract_license_from_readme(readme_text_preview)
        if license_rule:
            result['value'] = license_rule
            result['evidence_quote'] = license_evidence
            result['source_heading'] = 'regex_rule'
            result['evidence_type'] = 'readme_rule'
            result['confidence'] = max(float(result.get('confidence') or 0.0), 0.95)

    if property_name == 'description' and isinstance(result, dict):
        if not result.get('evidence_quote'):
            result['evidence_quote'] = _extract_description_evidence_quote_from_chunks(
                all_chunks,
                result.get('value'),
            )
        if result.get('evidence_quote'):
            result['evidence_type'] = 'readme_evidence'
            result['confidence'] = max(float(result.get('confidence') or 0.0), 0.6)

    return result

In [85]:
license_result = run_single_property('license')
print(json.dumps(license_result, indent=2, ensure_ascii=False))

{
  "repository_url": "https://github.com/CERN/TIGRE",
  "property": "license",
  "value": "BSD-3-Clause",
  "evidence_quote": "It is released under the BSD License, meaning you can use and modify the software freely.",
  "source_heading": "Licensing",
  "evidence_type": "readme_evidence",
  "confidence": 1.0,
  "retrieved_chunk_ids": [
    10,
    11,
    18,
    1
  ],
  "retrieved_headings": [
    "Licensing",
    "Licensing",
    "Contributors",
    "TIGRE: Tomographic Iterative GPU-based Reconstruction Toolbox"
  ],
  "retrieved_scores": [
    0.6843735575675964,
    0.5801926255226135,
    0.5359349250793457,
    0.5181072354316711
  ]
}


In [86]:
installation_result = run_single_property('installation')
print(json.dumps(installation_result, indent=2, ensure_ascii=False))

{
  "repository_url": "https://github.com/CERN/TIGRE",
  "property": "installation",
  "value": "MATLAB and Python builds are both fully supported.",
  "evidence_quote": "- [Installation instructions and requirements for MATLAB](Frontispiece/MATLAB_installation.md).<br>- [Installation instructions and requirements for Python](Frontispiece/python_installation.md).",
  "source_heading": "Installation",
  "evidence_type": "readme_evidence",
  "confidence": 1.0,
  "retrieved_chunk_ids": [
    4,
    1,
    5,
    15,
    10
  ],
  "retrieved_headings": [
    "Installation",
    "TIGRE: Tomographic Iterative GPU-based Reconstruction Toolbox",
    "Getting started",
    "Contributors",
    "Licensing"
  ],
  "retrieved_scores": [
    0.7466343641281128,
    0.5163472294807434,
    0.47413817048072815,
    0.42984437942504883,
    0.42738038301467896
  ]
}


In [87]:
contact_result = run_single_property('contact')
print(json.dumps(contact_result, indent=2, ensure_ascii=False))

{
  "repository_url": "https://github.com/CERN/TIGRE",
  "property": "contact",
  "value": "tigre.toolbox@gmail.com, ander.biguri@gmail.com",
  "evidence_quote": "Contact the authors directly at:\n\n[tigre.toolbox@gmail.com](mailto:tigre.toolbox@gmail.com) or [ander.biguri@gmail.com](mailto:ander.biguri@gmail.com)",
  "source_heading": "Contact",
  "evidence_type": "readme_evidence",
  "confidence": 1.0,
  "retrieved_chunk_ids": [
    9,
    17,
    12,
    13,
    16
  ],
  "retrieved_headings": [
    "Contact",
    "Contributors",
    "Contributors",
    "Contributors",
    "Contributors"
  ],
  "retrieved_scores": [
    0.775506854057312,
    0.5279214382171631,
    0.5151097774505615,
    0.514689028263092,
    0.5098159909248352
  ]
}


In [88]:
contributors_result = run_single_property('contributors')
print(json.dumps(contributors_result, indent=2, ensure_ascii=False))

{
  "repository_url": "https://github.com/CERN/TIGRE",
  "property": "contributors",
  "value": [
    {
      "name": "AnderBiguri",
      "github_url": "https://github.com/AnderBiguri"
    },
    {
      "name": "TIGRE",
      "github_url": "https://github.com/CERN/TIGRE"
    },
    {
      "name": "yliu88au",
      "github_url": "https://github.com/yliu88au"
    },
    {
      "name": "zezisme",
      "github_url": "https://github.com/zezisme"
    },
    {
      "name": "Pawelip",
      "github_url": "https://github.com/Pawelip"
    },
    {
      "name": "phernst",
      "github_url": "https://github.com/phernst"
    },
    {
      "name": "malena-sabate",
      "github_url": "https://github.com/malena-sabate"
    },
    {
      "name": "reubenlindroos",
      "github_url": "https://github.com/reubenlindroos"
    },
    {
      "name": "genusn",
      "github_url": "https://github.com/genusn"
    },
    {
      "name": "Daveelvt",
      "github_url": "https://github.com/Daveelvt"
  

In [89]:
links_result = run_single_property('links')
print(json.dumps(links_result, indent=2, ensure_ascii=False))

{
  "repository_url": "https://github.com/CERN/TIGRE",
  "property": "links",
  "value": [
    {
      "title": null,
      "url": "https://doi.org/10.1016/j.jpdc.2020.07.004",
      "relevance": "paper",
      "is_working": true,
      "status_code": 200,
      "final_url": "https://linkinghub.elsevier.com/retrieve/pii/S0743731520303336",
      "error": null
    },
    {
      "title": null,
      "url": "https://arxiv.org/abs/1905.03748",
      "relevance": "paper",
      "is_working": true,
      "status_code": 200,
      "final_url": "https://arxiv.org/abs/1905.03748",
      "error": null
    },
    {
      "title": null,
      "url": "https://github.com/CERN/TIGRE",
      "relevance": "paper",
      "is_working": true,
      "status_code": 200,
      "final_url": "https://github.com/CERN/TIGRE",
      "error": null
    },
    {
      "title": null,
      "url": "http://iopscience.iop.org/article/10.1088/2057-1976/2/5/055010",
      "relevance": "paper",
      "is_working": true,
 

In [90]:
description_result = run_single_property('description')
print(json.dumps(description_result, indent=2, ensure_ascii=False))

{
  "repository_url": "https://github.com/CERN/TIGRE",
  "property": "description",
  "value": "TIGRE is a GPU-based CT reconstruction software repository that contains a wide variety of iterative algorithms for high-performance x-ray absorption tomographic reconstruction.",
  "evidence_quote": "TIGRE features",
  "source_heading": "TIGRE features",
  "evidence_type": "readme_evidence",
  "confidence": 1.0,
  "retrieved_chunk_ids": [
    2,
    9,
    0,
    5,
    3,
    1
  ],
  "retrieved_headings": [
    "TIGRE features",
    "Contact",
    "TIGRE: Tomographic Iterative GPU-based Reconstruction Toolbox",
    "Getting started",
    "TIGRE features",
    "TIGRE: Tomographic Iterative GPU-based Reconstruction Toolbox"
  ],
  "retrieved_scores": [
    0.6716367602348328,
    0.552619457244873,
    0.5496375560760498,
    0.5377729535102844,
    0.5321694612503052,
    0.5269309878349304
  ]
}
